# Silent Diabetes: a no-blood-test screening tool

**Track:** AI + Health. Veritas AI International High School Hackathon 2026

**The idea:** about 64% of adults with diabetes in the UAE don't know they have it (IDF estimate). A pharmacist or a mall health-booth volunteer could ask about 10 simple questions (no blood test) and send higher-risk people to a doctor for an **HbA1c** blood test.

**This notebook:** loads the data, builds two baselines (including the American Diabetes Association's risk test), trains two models, picks a screening threshold, checks fairness, explains predictions, and saves the model for the app.

> This project is a prototype and is not medical advice.

## 0. Setup: install and import packages

Google Colab already has pandas, scikit-learn and matplotlib. We only need to install:
- `ucimlrepo`: downloads datasets from the UCI Machine Learning Repository
- `gradio`: builds the simple web app at the end

In [ ]:
# Install the two packages Colab doesn't have (-q = quiet, less text)
!pip install -q ucimlrepo gradio

In [ ]:
import pandas as pd               # tables of data
import numpy as np                # maths on arrays of numbers
import matplotlib.pyplot as plt   # charts

# One fixed 'random seed' so we get the same results every time we run the notebook
RANDOM_SEED = 42

print('Packages loaded.')

## 1. Load the data

**Dataset:** CDC Diabetes Health Indicators. These are answers from the 2015 BRFSS phone survey of **253,680 US adults**. Each row is one person.

We download it fresh every time (it is **not** stored in our GitHub repo). If the `ucimlrepo` package fails, we read the same CSV file directly from the UCI website instead.

In [ ]:
try:
    from ucimlrepo import fetch_ucirepo
    cdc = fetch_ucirepo(id=891)
    # cdc.data.features = the answers, cdc.data.targets = diabetes yes/no
    df = pd.concat([cdc.data.features, cdc.data.targets], axis=1)
    variable_info = cdc.variables   # official description of every column
    print('Loaded with ucimlrepo.')
except Exception as error:
    print('ucimlrepo failed:', error)
    print('Reading the CSV straight from the UCI website instead...')
    df = pd.read_csv('https://archive.ics.uci.edu/static/public/891/data.csv')
    df = df.drop(columns=['ID'])   # ID is just a row number, not useful
    variable_info = None

print('Rows (people):', len(df))
print('Columns:', len(df.columns))
df.head()

### 1a. Choose only the questions a pharmacist can ask

The dataset has 21 questions. We keep only the ones you can answer **without a lab test** and that are quick to ask at a counter:

| Column | Question we would ask | Codes in the data |
|---|---|---|
| `Age` | How old are you? | **Groups, not years:** 1 = 18-24, 2 = 25-29, ... 5-year steps ... 12 = 75-79, 13 = 80+ |
| `Sex` | Sex | 0 = female, 1 = male |
| `BMI` | Height and weight (we calculate BMI) | a number, e.g. 27 |
| `HighBP` | Has a doctor ever told you that you have high blood pressure? | 0 = no, 1 = yes |
| `HighChol` | ...high cholesterol? | 0 = no, 1 = yes |
| `PhysActivity` | Any exercise or physical activity in the last 30 days (not counting your job)? | 0 = no, 1 = yes |
| `GenHlth` | In general, how is your health? | 1 = excellent, 2 = very good, 3 = good, 4 = fair, 5 = poor |
| `DiffWalk` | Do you have serious difficulty walking or climbing stairs? | 0 = no, 1 = yes |
| `Smoker` | Have you smoked at least 100 cigarettes in your whole life? | 0 = no, 1 = yes |
| `HeartDiseaseorAttack` | Ever had heart disease or a heart attack? | 0 = no, 1 = yes |
| `Stroke` | Ever had a stroke? | 0 = no, 1 = yes |

**Left out on purpose:** `CholCheck`, `AnyHealthcare`, `NoDocbcCost`, `Income`, `Education`, mental/physical-health days, diet and alcohol questions. They are either about the healthcare system, too personal for a mall booth, or slow to answer.

The next cell prints the official description of each column so we can check the table above against the source.

In [ ]:
FEATURES = ['Age', 'Sex', 'BMI', 'HighBP', 'HighChol', 'PhysActivity',
            'GenHlth', 'DiffWalk', 'Smoker', 'HeartDiseaseorAttack', 'Stroke']
TARGET = 'Diabetes_binary'

# Show the official description of each column we use
if variable_info is not None:
    pd.set_option('display.max_colwidth', None)
    display(variable_info[variable_info['name'].isin(FEATURES + [TARGET])][['name', 'description']])

# Keep only our columns
df = df[FEATURES + [TARGET]]

# Check the range of values in each column matches the codes in the table
df.describe().loc[['min', 'max', 'mean']].round(2)

### 1b. What does 'diabetes = 1' really mean? (prediabetes check)

The official description says `Diabetes_binary`: **0 = no diabetes, 1 = prediabetes or diabetes**. We checked whether that's true by counting.

The same dataset also exists as a 3-class version on Kaggle (`diabetes_012_health_indicators_BRFSS2015.csv`). Its published counts are:
- 0 = no diabetes: **213,703**
- 1 = prediabetes: **4,631**
- 2 = diabetes: **35,346**

If our file's '1' really included prediabetes, we would see 4,631 + 35,346 = 39,977 people with '1'. Run the next cell to see the real count.

In [ ]:
counts = df[TARGET].value_counts().sort_index()
print(counts)
print()
print('People with diabetes = 1:', counts[1])
print('Matches the diabetes-only count (35,346)?        ', counts[1] == 35346)
print('Matches prediabetes + diabetes (39,977)?         ', counts[1] == 4631 + 35346)
print('Does 0 match no-diabetes + prediabetes (218,334)?', counts[0] == 213703 + 4631)

**Result and our decision:** the count is 35,346, exactly the diabetes-only number. So even though the description says otherwise, **the 4,631 prediabetic people were put in the '0' group.**

What we do about it:
- Our target means **'has diabetes' (diagnosed)**. We say this honestly everywhere.
- About 2% of the 'no diabetes' group actually has prediabetes. This makes the model slightly *more cautious* than it should be, because some people it flags as 'risky' are labelled 'no' even though they are on the way to diabetes. We list this in Limitations.
- For the real tool this is fine: if someone is flagged, the HbA1c test will catch both diabetes and prediabetes.

## 2. Explore the data

### 2a. Class balance and missing values

In [ ]:
print('Missing values in each column:')
print(df.isna().sum())
print()

share = df[TARGET].mean()
print(f'Share of people with diabetes: {share:.1%}')
print(f'Share without diabetes:        {1 - share:.1%}')
print()
print('Exact duplicate rows:', df.duplicated().sum())

**What this means:**
- **No missing values.** The dataset was already cleaned by its creators.
- Only about **14%** of people have diabetes. The data is **imbalanced**. A lazy model could say 'no' to everyone and still be about 86% 'accurate'. We'll prove this in Baseline 1.
- **Duplicate rows:** with only 11 simple questions, many different people give exactly the same answers (e.g. two healthy 30-year-old women with BMI 24). These are real, different people, so **we keep them**. We just have to make sure our test results aren't fooled by this, so we'll check for it later.

### 2b. Charts: how does diabetes risk change with each answer?

In [ ]:
# Labels for the 13 age groups, so the chart is readable
AGE_LABELS = {1: '18-24', 2: '25-29', 3: '30-34', 4: '35-39', 5: '40-44', 6: '45-49',
              7: '50-54', 8: '55-59', 9: '60-64', 10: '65-69', 11: '70-74', 12: '75-79', 13: '80+'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Chart 1: % with diabetes in each age group
by_age = df.groupby('Age')[TARGET].mean() * 100
axes[0].bar([AGE_LABELS[a] for a in by_age.index], by_age.values, color='tab:orange')
axes[0].set_title('% with diabetes, by age group')
axes[0].set_ylabel('% with diabetes')
axes[0].tick_params(axis='x', rotation=60)

# Chart 2: BMI for people with and without diabetes
axes[1].hist(df[df[TARGET] == 0]['BMI'], bins=range(12, 60), alpha=0.6, density=True, label='No diabetes')
axes[1].hist(df[df[TARGET] == 1]['BMI'], bins=range(12, 60), alpha=0.6, density=True, label='Diabetes')
axes[1].set_title('BMI distribution')
axes[1].set_xlabel('BMI')
axes[1].legend()

# Chart 3: % with diabetes by self-rated general health
by_health = df.groupby('GenHlth')[TARGET].mean() * 100
health_names = ['Excellent', 'Very good', 'Good', 'Fair', 'Poor']
axes[2].bar(health_names, by_health.values, color='tab:orange')
axes[2].set_title('% with diabetes, by general health')
axes[2].set_ylabel('% with diabetes')

plt.tight_layout()
plt.show()

In [ ]:
# For each yes/no question: % with diabetes among people who said NO vs YES
yes_no_questions = ['HighBP', 'HighChol', 'PhysActivity', 'DiffWalk', 'Smoker',
                    'HeartDiseaseorAttack', 'Stroke', 'Sex']

table = pd.DataFrame({
    'answer = 0 (no / female)': [df[df[q] == 0][TARGET].mean() * 100 for q in yes_no_questions],
    'answer = 1 (yes / male)':  [df[df[q] == 1][TARGET].mean() * 100 for q in yes_no_questions],
}, index=yes_no_questions).round(1)

print('% of people with diabetes, depending on their answer:')
table

**What the charts show:**
- **Age:** risk climbs steadily with age, from almost nobody at 18-24 to around 1 in 5 people in their 70s.
- **BMI:** the diabetes group is shifted toward higher BMI, but the two groups overlap a lot. BMI alone can't separate them.
- **General health:** people who say their health is 'poor' have diabetes far more often than people who say 'excellent'. This is a strong signal, but partly because having diabetes *makes* people rate their health lower (cause and effect go both ways).
- **Yes/no questions:** high blood pressure, high cholesterol, heart disease and difficulty walking are each linked to much higher diabetes rates. Being physically active is linked to *lower* rates.

No single question is enough on its own. That's why we combine them in a model.